# Consulta 4, 5 y 6

In [1]:
import pathlib
import csv
import json

#### 4. Los aeropuertos, lagos y tipo de conectividad en provincias con población mayor o menor a un valor que se pueda especificar fácilmente.

>   seek_lakes() recibe el nombre de la provincia para buscar respecto a ella todos sus lagos. Creo una lista vacía para guardar el nombre de los lagos encontrados. Abro el archivo con with porque es seguro y limpio, en lakes_reader guardo el iterador de lagos_arg.csv, con next(lake_reader) me salteo el header para acceder ya a la información importante.

>   dentro del bucle for se itera por cada linea y pregunta si la provincia actual, row[1], es la provincia recibida, si es así agrego con .append() a la lista vacía, pero agrego el nombre del lago, row[0], que es lo que importa guardar. 

In [29]:
def seek_lakes(province_name):
    lakes =[]   # lista de lagos
    path_lagos = pathlib.Path('..','data_modify','lagos_arg.csv')
    with open(path_lagos,'r',encoding='utf-8',newline='') as lakes_file:
        lakes_reader = csv.reader(lakes_file)
        next(lakes_reader)  # salteo el header

        for row in lakes_reader:
            if row[1]==province_name:   # row[1] contiene el nombre de la provincia dentro del archivo
                lakes.append(row[0])    # row[0] contiene el nombre del lago que le pertenece a la provincia
    return lakes

>   seek_airports() recibe el nombre de la provincia para buscar respecto a ella todos sus aeropuertos. Creo una lista vacía para guardar el nombre encontrados. Abro el archivo con with porque es seguro y limpio, en airports_reader guardo el iterador de ar-airports.csv, con next(airports_reader) me salteo el header para acceder ya a la información importante.

>   dentro del bucle for se itera por cada linea y pregunta si la provincia actual, row[24], es la provincia recibida, si es así agrego con .append() a la lista vacía, pero agrego el nombre del aeropuerto, row[3], que es lo que importa guardar. 

In [30]:
def seek_airports(province_name):
    airports = []   # lista vacía para guardar los nombres de los aeropuertos
    path_airports = pathlib.Path('..','data_modify','ar-airports.csv')
    with open(path_airports,'r',encoding='utf-8',newline='') as airports_file:
        airports_reader = csv.reader(airports_file)
        next(airports_file)

        for row in airports_reader:
            if row[24]==province_name:
                airports.append(row[3])
    return airports

>   seek_connectivity() recibe el nombre de la provincia para buscar respecto a ella los tipos de conectividad que tiene. Creo un conjunto vacío para guardar los tipos de conectividad, no uso una lista porque pueden repetirse y con set() me ahorro esos repetidos. Abro el archivo con with porque es seguro y limpio, en connectivity_reader guardo el iterador de Conectividad_Internet.csv, y en header guardo el encabezado con next(connectivity_reader). Es necesario el encabezado pora conocer los tipos de conectividad

>   dentro del bucle for itero por cada linea y pregunto si se trata de la provincia solicitada, si es así entonces preguntó primero si por lo menos tiene conectividad, en caso de que no tenga salto a la siguiente linea, en caso de tenerla se inicia un bucle for que itera sólamente entre las columnas 5 y 13, me guardo el indice con enumerate() y en cada celda pregunto si tiene conectividad de ese tipo, en caso de tenerla guardo el tipo de conectividad en el conjunto connectivity. A la hora de rotornar, json no puede imprimir objetos de tipo set así que lo transformo en lista.

In [35]:
def seek_connectivity(province_name):
    connectivity = set()    # conjunto para guardar tipos de conexión sin repetidos
    path_connectivity = pathlib.Path('..','data_modify','Conectividad_Internet.csv')
    with open(path_connectivity,'r',encoding='utf-8',newline='') as connectivity_file:
        connectivity_reader = csv.reader(connectivity_file)
        header = next(connectivity_reader)  # guardo el encabezado

        for row in connectivity_reader:
            if row[0].lower()==province_name:   # row[0] contiene el nombre de la provincia
                if row[16]=='SI':   # row[16] contiene 'NO' en caso de no haber ninguna conectividad
                    for i,cell in enumerate(row[4:13]):     # los tipos se encuentran en este rango
                        if cell=='SI':      # tiene este tipo de conexión
                            connectivity.add(header[i +4])   
    return list(connectivity)   # transformo el set en list

>   seek_provinces() recibe un diccionario, un entero y un string. Abro el archivo del censo (porque contiene un resumen de la población por provincia y no por partidos) con with por seguridad, en c2022_reader guardo el iterador del archivo y me salteo el encabezado y la fila siguiente porque es un resumen del pais entero.

>   independientemente de si el usuario eligió menor o mayor población, el proceso será. El bucle for itera por filas en el archivo csv, si la población cumple con la condición que sea menor o mayor, se agrega la provincia al diccionario. 

>   En el diccionario encontramos tres claves: lakes, airports, y connnectivity. Cada clave llama a una función respecto a lo que busca, cada una necesita manejar archivos diferentes y como parámetro reciben el nombre de la provincia (row[0]).

In [36]:
def seek_provinces (provinces,population,aux):
    path = pathlib.Path('..','data_modify','c2022_tp_c_resumen_adaptado.csv')
    with open(path, 'r',encoding='utf-8',newline='') as c2022_file:
        c2022_reader = csv.reader(c2022_file)
        next(c2022_reader)  # salteo el encabezado
        next(c2022_reader)  # salteo la primera fila del total del pais
        
        if aux=='menor':    # si el usuario eligió menor...
            for row in c2022_reader:
                if (int(row[1])<population):    # busca por menor población
                                                # row[1] contiene la población total de la provincia
                    provinces[row[0]] = {       # row[0] contiene el nombre de la provincia
                        'lakes':seek_lakes(row[0]),
                        'airports':seek_airports(row[0]),
                        'connectivity':seek_connectivity(row[0].lower()),
                        }
        else:
            for row in c2022_reader:
                if (int(row[1])>population):    # busca por mayor población

                    provinces[row[0]] = {
                        'lakes':seek_lakes(row[0]),
                        'airports':seek_airports(row[0]),
                        'connectivity':seek_connectivity(row[0].lower()),
                        }

>   En population guardo la cantidad ingresada por el usuario transformada en entero. Mientras el usuario no ingrese el valor 'menor' o 'mayor' el bucle while va a seguir pidiendo que se ingrese un valor, una vez que la condición del if se cumple, el bucle while se rompe. Luego se llama a la función seek_provinces() que recibe un diccionario destinado a guardar como clave los nombres de las provincias que, a su vez, guardan un diccionario con más claves, population y aux que contiene 'menor' o 'mayor'

In [ ]:
provinces = {}
population = int(input('Ingrese una cantidad de población: '))

while True:
    aux = input('elija entre menor/mayor población para buscar')
    if (aux=='menor') or (aux=='mayor'):
        break
    else:
        print('Por favor, ingrese "menor" o "mayor"')

seek_provinces(provinces,population,aux)
print(json.dumps(provinces, indent=4))

#### 5. Mostrar los aeropuertos en las capitales de cada provincia.

>   Defino CAPITALES como una lista de todas las capitales de Argentina

In [43]:
CAPITALES = ['La Plata', 'Catamarca','Resistencia','Rawson','Córdoba','Corrientes','Paraná','Formosa','San Salvador de Jujuy','Santa Rosa','La Rioja','Mendoza','Posadas','Neuquén','Viedma','Salta','San Juan','San Luis','Río Gallegos','Santa Fe','Santiago del Estero','Ushuaia','San Miguel de Tucumán']

>   Abro el archivo ar-airports.csv, guardo el iterador y salteo el encabezado.

>   Utilizo la función filter con lambda para quedarme sólo con las filas del archivo que tienen un aeropuerto en una capital y la información la guardo en airports_cities.

>   airports_cities_dict es un diccionario donde cada clave es una capital y cada valor en una lista de aeropuertos. En el bucle for itero por lista en airports_cities. Si la capital, line[13], se encuentra en airports_cities_dict entonces agrego el aeropuerto, line[3]. En caso de no existir la capital, creo la clave y el valor será una lista con un solo elemento, el aeropuerto actual.

In [ ]:
airports_cities_dict = {}   # dict pensado para clave: capital y valor: lista de aeropuertos
path_airports = pathlib.Path('..','data_modify','ar-airports.csv')
with open(path_airports,'r',encoding='utf-8',newline='') as airports_file:
    airports_reader = csv.reader(airports_file)
    next(airports_reader)   # salteo el encabezado

    airports_cities = filter(lambda row: row[13] in CAPITALES, airports_reader) # me quedo sólo con las capitales
    
    for line in airports_cities:
        # si la capital está en el dict, agrego un aeropuerto más en la lista
        if line[13] in airports_cities_dict:   
            airports_cities_dict[line[13]].append(line[3])
        
        # sino, creo la clave: capital y valor: lista con el primer aeropuerto encontrado de la capital
        else:
            airports_cities_dict[line[13]] = [line[3]]
    
print(json.dumps(airports_cities_dict, indent=4))

#### 6. Mostrar los Lagos de una superficie según la columna 'Sup Tamaño' donde el criterio (chico, medio, grande) se puede indicar fácilmente.

>   Creo tres listas vacías para los tres tipos de superficie de lagos. Abro el archivo lagos_arg.csv, guardo el iterador y salteo el header que no necesito procesar. 

>   En el bucle for itero por cada fila del archivo, en caso de que la columna 'Sub Tamaño' tenga el valor grande, agrego el lago actual en la lista big; si el valor es medio, agrego el lago en la lista medium; en caso que no sea grande ni mediano se agrega a la lista small porque se toma que el valor es chico. 

>   En pantalla se muestra un menú de opciones para el usuario, debe elegir entre las opciones 1, 2 o 3 dependiendo qué información desea ver.

In [ ]:
big = []        # lista para los lagos grandes
medium = []     # lista para los lagos medianos 
small = []      # lista para los lagos chicos

path_lakes = pathlib.Path('..','data_modify','lagos_arg.csv')
with open(path_lakes,'r',encoding='utf-8',newline='') as lakes_file:
    lakes_reader = csv.reader(lakes_file)
    next(lakes_reader)      # salteo el encabezado

    for row in lakes_reader:
        if row[8] == 'grande':      # row[8] contiene el valor de 'Sub Tamaño'
            big.append(row[0])      # row[0] contiene el nombre del lago
        elif row[8] == 'medio':
            medium.append(row[0])
        else:
            small.append(row[0])

# menú
print('''
    Lagos con superficies grandes: 1
    Lagos con superficies medianas: 2
    Lagos con superficies chicas: 3
''')
aux = int(input('Opción: '))        # el usuario indica qué lagos de qué superficie necesita ver

if aux==1:
    print(json.dumps(big, indent=2))
elif aux==2:
    print(json.dumps(medium, indent=2))
else:
    print(json.dumps(small, indent=2))